# 08  Land Registry property signals

This notebook answers one question: **which of the companies in our list own commercial property in
England and Wales?** Owning property is a useful commercial signal. It shows a firm has an asset
base, and a company with property is a natural fit for a commercial mortgage, refinancing or a
lending conversation.

For each company we can match, we save one `owns_property` record into our shared results table
(`signals`), tagged with the company's Companies House number so the rest of the team can join it to
the main dataset later. This notebook only produces those records; it does not change anyone else's
work.

## About the data source: HM Land Registry (CCOD)

The data comes from **HM Land Registry's "UK companies that own property in England and Wales"**
dataset, still known by its old name **CCOD** (Commercial and Corporate Ownership Data). It is
published through the government's *Use Land and Property Data* service.

**How we get it**
- It is **free**, but **licensed**: you create a free account, accept the CCOD licence once, and you
  are given a personal **API key**.
- This notebook calls the Land Registry API (`https://use-land-property-data.service.gov.uk/api/v1`)
  with that key, finds the latest monthly file, and downloads it. You paste the key in when asked; it
  is never saved anywhere.
- The file is refreshed **monthly** and is a full snapshot each time (about 1.5 GB unzipped).

**What is in it**
- One row per **property title**. Each title can list up to **four owners**.
- For each owner: name, **company registration number** (where available), owner type, and address.
- For each property: address, region, postcode, tenure (freehold or leasehold), price paid, and the
  date the owner was added.

**Limitations to be honest about**
- **England and Wales only.** A company registered in Scotland or Northern Ireland appears only if it
  owns property in England or Wales, so this signal under-reaches those firms.
- The **registration number is "where available"**. Land Registry warns it can contain typing errors,
  and property bought **before 1996** may carry no number at all.
- The postcode column is the **property's** postcode. The **owner's** postcode sits inside the owner
  address text, so we read it out of there when we need it.
- The data is **licensed to your account**, so we do **not** commit the file to GitHub; it is simply
  re-downloaded when needed.

## How to run
Run NB05 first so `lloyds.duckdb` exists. Then run this notebook top to bottom; it updates the same
database. In Colab it reads and writes everything in your `MyDrive/Lloyds` folder.

## 1. Install the tools we need

This box makes sure the two extra tools are available, installing them only if they are missing.

In [ ]:
import sys, subprocess
# Colab does not always ship these two, so install them if the import fails:
#   duckdb    = a small local database that holds our companies and results
#   rapidfuzz = fast "close enough" text matching, used to compare company names
for pkg in ["duckdb", "rapidfuzz"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

This box switches the tools on and prints their versions, so we can confirm they loaded.

In [ ]:
import duckdb                              # the local database that stores companies and results
import pandas as pd                        # tables and data handling
import requests                            # downloads the Land Registry file over the web
import re                                  # text pattern matching (cleaning names, reading postcodes)
from collections import defaultdict        # handy dictionaries for the quick lookups we build later
from datetime import datetime, timezone    # to timestamp when we pulled the data

print("duckdb", duckdb.__version__, "| pandas", pd.__version__)

## 2. Find the database and get the Land Registry file
We need two things: the `lloyds.duckdb` database that NB05 built (it holds the company list and the
results table), and the Land Registry file itself. You have an API key, so the notebook downloads the
file for you.

This box finds the database, then downloads the latest Land Registry file using your API key. If you leave the key prompt blank, it instead uses any CCOD file you have already saved in the folder. `CCOD_MAX_ROWS` is a dial: leave it as `None` for the whole file, or set a number like 200000 for a quick trial run.

In [ ]:
from pathlib import Path

# Work out whether we are running in Colab or on a local machine, and set the folders to match.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/Lloyds")     # same Drive folder as NB05
    DB_PATH = WORK_DIR / "lloyds.duckdb"
else:
    WORK_DIR = Path("..").resolve() / "data" / "processed"
    DB_PATH = WORK_DIR / "lloyds.duckdb"

# The database must already exist. If not, NB05 has not been run yet.
assert DB_PATH.exists(), f"lloyds.duckdb not found at {DB_PATH}. Run NB05 first."
print("DB:", DB_PATH)

CCOD_MAX_ROWS = None        # None = read the whole file; a number = read only that many rows (faster)
NOW = datetime.now(timezone.utc).isoformat(timespec="seconds")   # timestamp saved with each result

# --- download the Land Registry file with your API key ---------------------------------------------
# The API needs your key on every request. It first lists the available files, then gives a short-lived
# link to download the newest "FULL" one. The file is a zip, so we unzip the CSV inside it.
import io, zipfile, getpass
API_BASE = "https://use-land-property-data.service.gov.uk/api/v1"

def download_ccod_via_api(work_dir):
    key = getpass.getpass("Land Registry API key (blank = use a file already in the folder): ").strip()
    if not key:
        return None                                     # no key given -> fall back to a local file
    hdr = {"Authorization": key, "Accept": "application/json"}

    # 1) ask the API which files exist for the CCOD dataset
    meta = requests.get(f"{API_BASE}/datasets/ccod", headers=hdr, timeout=60).json()
    resources = meta.get("result", {}).get("resources", []) if isinstance(meta.get("result"), dict) else []
    # 2) pick the newest "FULL" file (a full monthly snapshot, not the smaller "change only" file)
    fulls = sorted(r.get("file_name", "") for r in resources if "FULL" in r.get("file_name", "").upper())
    if not fulls:
        raise RuntimeError(f"API listed no FULL file. Full response: {meta}")
    fname = fulls[-1]
    # 3) ask for a download link for that file
    link = requests.get(f"{API_BASE}/datasets/ccod/{fname}", headers=hdr, timeout=60).json()
    url = (link.get("result") or {}).get("download_url")
    if not url:
        raise RuntimeError(f"No download_url for {fname}. Full response: {link}")
    # 4) download it, and unzip the CSV inside
    print("downloading", fname, "...")
    blob = requests.get(url, timeout=600).content
    if fname.lower().endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(blob)) as z:
            csv_name = next(n for n in z.namelist() if n.lower().endswith(".csv"))
            z.extract(csv_name, work_dir)
            return work_dir / csv_name
    out = work_dir / fname
    out.write_bytes(blob)
    return out

CCOD_PATH = download_ccod_via_api(WORK_DIR)
if CCOD_PATH is None:
    # no key entered: look for a CCOD file you downloaded earlier
    files = sorted(WORK_DIR.glob("CCOD*.csv")) + sorted(WORK_DIR.glob("ccod*.csv"))
    if not files:
        raise FileNotFoundError(
            f"No API key given and no CCOD csv found in {WORK_DIR}.\n"
            "Either paste your API key at the prompt, or download CCOD_FULL_YYYY_MM.csv from\n"
            "  https://use-land-property-data.service.gov.uk/datasets/ccod\n"
            f"and drop it into {WORK_DIR}.")
    CCOD_PATH = files[-1]
print("Land Registry file:", CCOD_PATH.name)

## 3. Small text helpers
Before we can match names, we tidy them into a standard form, and we add a reader that pulls a postcode out of an address line. These are the same helpers used across all the signal notebooks, so a company is matched the same way everywhere.

This box builds three small tools: one tidies a company number, one tidies a company name (so "A.B.C. Limited" and "ABC LTD" line up), and one reads a UK postcode out of free address text.

In [ ]:
def clean_company_number(value):
    # Standardise a Companies House number so the same company always looks the same.
    if value is None:
        return None
    s = str(value).strip().upper()
    if s == "" or s == "NAN":
        return None
    if s.isdigit():
        return s.zfill(8)     # plain numbers are padded to 8 digits, e.g. "1234567" -> "01234567"
    return s                  # letter-prefixed numbers (OC, SC, NI, FC ...) are left as they are

# Company names carry lots of noise (punctuation, "LIMITED", "THE", "AND"). We strip that out so two
# ways of writing the same firm end up identical.
_SUFFIXES = [
    "LIMITED", "LTD", "PLC", "PUBLIC LIMITED COMPANY", "LLP",
    "LIMITED LIABILITY PARTNERSHIP", "LP", "CIC", "CIO",
    "COMPANY", "CO", "AND", "THE",
]
_SUFFIX_RE = re.compile(r"\b(" + "|".join(_SUFFIXES) + r")\b")

def normalise_name(name):
    if name is None:
        return None
    s = str(name).upper()
    s = re.sub(r"[^A-Z0-9 ]", " ", s)     # drop punctuation, keep letters, numbers and spaces
    s = _SUFFIX_RE.sub(" ", s)            # remove the common company-type words
    s = re.sub(r"\s+", " ", s).strip()    # collapse repeated spaces
    return s or None

def normalise_postcode(pc):
    # A postcode in a comparable form: uppercase, no spaces. "sw1a 1aa" -> "SW1A1AA".
    if pc is None:
        return None
    s = re.sub(r"\s+", "", str(pc).upper())
    return s or None

# An owner's address in the Land Registry file is free text, e.g.
# "Sixth Floor, 6 Chesterfield Gardens, London W1J 5BQ". This pulls the postcode out by its shape.
_UK_PC_RE = re.compile(r"\b([A-Z]{1,2}[0-9][A-Z0-9]?\s*[0-9][A-Z]{2})\b")
def extract_postcode(text):
    if text is None or (isinstance(text, float)):
        return None
    found = _UK_PC_RE.findall(str(text).upper())
    return found[-1] if found else None    # if several appear, take the last (usually the real one)

print("helpers ready")

## 4. Load our company list and build quick lookups
We read every company from the database (its number, tidied name, and postcode), then build two
lookups so matching stays fast even on hundreds of thousands of companies:
- an **exact-name lookup** (find a company instantly by its tidied name), and
- **first-letter-word blocks** (group companies by the first word of their name, so the fuzzy step
  only compares names that start the same way).

This box reads the master list of companies and builds the two fast lookups described above. It prints how many companies were loaded so you can confirm the full list is in use.

In [ ]:
con = duckdb.connect(str(DB_PATH))
# company_list = every company we might match against (number, tidied name, postcode)
company_list = con.execute("SELECT company_number, name_norm, postcode FROM companies").df()
print(f"companies in our list: {len(company_list):,}")

known_company_numbers = set(company_list["company_number"])   # for the instant number check
by_name = defaultdict(list)     # tidied name        -> [(company_number, postcode), ...]
blocks  = defaultdict(list)     # first word of name -> [(tidied name, company_number, postcode), ...]
for cn, nm, pc in zip(company_list["company_number"], company_list["name_norm"], company_list["postcode"]):
    if not nm:
        continue
    pcn = normalise_postcode(pc)
    by_name[nm].append((cn, pcn))
    blocks[nm.split(" ")[0]].append((nm, cn, pcn))
print(f"exact-name entries: {len(by_name):,} | first-word groups: {len(blocks):,}")

## 5. How we decide a match (the matching ladder)
Most Land Registry rows carry a Companies House number, which is an exact, reliable match. For the
rows that only give a name, we fall back to matching by name, but only if the **postcode also agrees**,
so a small firm is never confused with a big national company that happens to share a name.

We try the methods strongest first, and record which one was used as a confidence score:
1. `company_number` (1.0) - the row carries a Companies House number that is in our list.
2. `name_exact_postcode` (0.95) - exact tidied name and the owner's postcode agrees.
3. `name_fuzzy_postcode` (~0.8-0.9) - a close (fuzzy) name and the postcode also agrees.
4. `name_exact_unconfirmed` (0.6) - exact name, but there was no postcode to confirm with.

This box writes the matching ladder as a function. For each owner it tries the reliable methods first and stops at the first one that works, returning the matched company, a confidence score, and the method used.

In [ ]:
FUZZY_CUTOFF = 90     # minimum "close enough" score (0-100) before we accept a fuzzy name match

def _postcode_match(a, b):
    return bool(a and b and a == b)      # both present and identical

def match_owner(name, reg_no, owner_postcode=None):
    # 1) exact Companies House number: the most reliable match, needs no postcode
    cn = clean_company_number(reg_no)
    if cn and cn in known_company_numbers:
        return cn, 1.0, "company_number"

    nm = normalise_name(name)
    if not nm:
        return None, 0.0, "no_match"
    own_pc = normalise_postcode(owner_postcode)

    # 2) exact tidied name, confirmed by postcode where we have one
    if nm in by_name:
        cands = by_name[nm]
        confirmed = [c for c, pc in cands if _postcode_match(own_pc, pc)]
        if confirmed:
            return confirmed[0], 0.95, "name_exact_postcode"
        if own_pc:
            # same name but the postcode does not match any of them: probably a different firm
            return None, 0.0, "name_exact_postcode_mismatch"
        if len(cands) == 1:
            return cands[0][0], 0.6, "name_exact_unconfirmed"   # one candidate, no postcode to check
        return None, 0.0, "name_exact_ambiguous"                # several same-name firms, cannot tell

    # 3) fuzzy name within the same first-word group, only when the postcode agrees too
    if own_pc:
        bucket = blocks.get(nm.split(" ")[0])
        if bucket:
            choices = [b[0] for b in bucket]
            for _, score, idx in process.extract(
                    nm, choices, scorer=fuzz.WRatio, score_cutoff=FUZZY_CUTOFF, limit=5):
                if _postcode_match(own_pc, bucket[idx][2]):
                    return bucket[idx][1], round(score / 100 * 0.9, 3), "name_fuzzy_postcode"

    return None, 0.0, "no_match"

# rapidfuzz is only needed for the fuzzy step; import it here next to where it is used
from rapidfuzz import process, fuzz

# quick check: a made-up number that is not in our list should not match
print(match_owner("SOME COMPANY LTD", "ZZ999999"))

## 6. Read the Land Registry file and tidy it into one row per owner
Each row in the file is one property and can name up to four owners spread across many columns. We
reshape it into a simple long list: **one row per owner**, keeping the property details, the owner's
name and registration number, and a postcode read from the owner's address.

This box reads the file (only the columns we need, to save memory) and unfolds the up-to-four-owners layout into a tidy list of one row per owner. It then prints how many owner rows we have and how many carry a registration number or a postcode, and shows the first three rows.

In [ ]:
# the columns that describe the property (shared by all owners on that title)
base_cols = ["Title Number", "Tenure", "Property Address", "Region",
             "Postcode", "Price Paid", "Date Proprietor Added"]
# the columns that repeat for each of the four possible owners
prop_cols = []
for i in (1, 2, 3, 4):
    prop_cols += [f"Proprietor Name ({i})", f"Company Registration No. ({i})",
                  f"Proprietorship Category ({i})",
                  f"Proprietor ({i}) Address (1)", f"Proprietor ({i}) Address (2)",
                  f"Proprietor ({i}) Address (3)"]
wanted = set(base_cols + prop_cols)

# read only the columns we need, everything as text, so the large file fits in Colab memory
raw = pd.read_csv(CCOD_PATH, dtype=str, usecols=lambda c: c in wanted,
                  nrows=CCOD_MAX_ROWS, na_values=[""], keep_default_na=False)
print(f"property titles in the file: {len(raw):,}")

# unfold: for each of the four owner slots, take the rows that have an owner there and stack them
frames = []
for i in (1, 2, 3, 4):
    name_col = f"Proprietor Name ({i})"
    if name_col not in raw.columns:
        continue
    sub = raw[raw[name_col].notna()].copy()
    if sub.empty:
        continue
    # join the owner's three address lines into one string, then read the postcode out of it
    owner_addr = (sub[f"Proprietor ({i}) Address (1)"].fillna("") + " "
                  + sub[f"Proprietor ({i}) Address (2)"].fillna("") + " "
                  + sub[f"Proprietor ({i}) Address (3)"].fillna(""))
    frames.append(pd.DataFrame({
        "title_number":            sub["Title Number"],
        "tenure":                  sub["Tenure"],                     # freehold or leasehold
        "property_address":        sub["Property Address"],
        "region":                  sub["Region"],
        "price_paid":              sub["Price Paid"],                 # blank on many rows
        "date_added":              sub["Date Proprietor Added"],
        "proprietor_name":         sub[name_col],                     # the owner (company) name
        "company_reg_no":          sub[f"Company Registration No. ({i})"],
        "proprietorship_category": sub[f"Proprietorship Category ({i})"],   # owner type
        "owner_postcode":          owner_addr.map(extract_postcode),
    }))

owners = pd.concat(frames, ignore_index=True)
print(f"owner rows (one company on one property): {len(owners):,}")
print(f"  with a registration number: {owners['company_reg_no'].notna().sum():,}")
print(f"  with a readable owner postcode: {owners['owner_postcode'].notna().sum():,}")
owners.head(3)

### A quick feel for the data
Before matching, it helps to see what kinds of owners are in the file and how the properties split between freehold and leasehold.

This box shows two small tables: the most common owner types, and the freehold/leasehold split.

In [ ]:
from IPython.display import display

print("Most common owner types:")
display(owners["proprietorship_category"].fillna("Unknown").value_counts().head(10).to_frame("owner rows"))

print("\nFreehold vs leasehold:")
owners["tenure"].fillna("Unknown").value_counts().to_frame("owner rows")

## 7. Match owners to our company list and save the property signals
First the fast, exact part: match on the registration number in one step, which handles most rows.
Then the smaller leftover set (owners with no usable registration number) goes through the name
ladder. Everything that matches a company in our list is written into the `signals` table as an
`owns_property` record. We clear any previous Land Registry records first so re-runs do not double
count.

This box does the matching in two parts (fast number match, then the name ladder for the rest), tidies the value and date, and saves the matches into the results table. It clears old Land Registry records first so re-running does not double count, then prints how many matched and by which method.

In [ ]:
# --- part 1: fast exact match on the registration number (handles most rows in one step) ----------
owners["clean_reg"] = owners["company_reg_no"].map(clean_company_number)
in_list = owners["clean_reg"].isin(known_company_numbers)   # True where the number is one of ours

reg_matched = owners[in_list].copy()
reg_matched["company_number"] = reg_matched["clean_reg"]
reg_matched["confidence"] = 1.0
reg_matched["method"] = "company_number"

# --- part 2: name ladder for the leftover rows only (keeps the whole file quick to process) --------
residual = owners[~in_list]
name_rows = []
for r in residual.itertuples(index=False):
    cn, conf, method = match_owner(r.proprietor_name, r.company_reg_no, r.owner_postcode)
    if cn:
        d = r._asdict()
        d.update(company_number=cn, confidence=conf, method=method)
        name_rows.append(d)
name_matched = pd.DataFrame(name_rows) if name_rows else pd.DataFrame(columns=reg_matched.columns)

matched = pd.concat([reg_matched, name_matched], ignore_index=True)

# tidy the fields we store: price as a number, date as a real date (the file uses day-month-year)
matched["value"] = pd.to_numeric(matched["price_paid"], errors="coerce")
matched["signal_date"] = pd.to_datetime(matched["date_added"], errors="coerce",
                                        dayfirst=True).dt.date
matched["detail"] = matched["property_address"].astype(str).str.slice(0, 200)   # keep it short
matched["signal_type"] = "owns_property"
matched["source"] = "land_registry_ccod"
matched["retrieved_at"] = NOW

print(f"owner rows matched to a company in our list: {len(matched):,} of {len(owners):,}")
print("\nby method:")
print(matched["method"].value_counts().to_string())

# clear our previous Land Registry rows, then insert the fresh ones (so re-runs never double count)
con.execute("DELETE FROM signals WHERE source = 'land_registry_ccod'")
ins = matched[["company_number", "signal_type", "signal_date", "value", "detail",
               "source", "confidence", "retrieved_at"]]
con.register("tmp_sig", ins)
con.execute("""INSERT INTO signals
               SELECT company_number, signal_type, signal_date, value, detail,
                      source, confidence, retrieved_at
               FROM tmp_sig""")
con.unregister("tmp_sig")
print(f"\nsaved to the results table: {len(ins):,} rows")

## 8. Results: how many companies own property
How many companies in our list now carry a property record, and who owns the most.

This box prints the headline coverage number and lists the ten companies that own the most properties, joined back to their real names.

In [ ]:
n_companies = con.execute(
    "SELECT count(DISTINCT company_number) FROM signals WHERE source='land_registry_ccod'"
).fetchone()[0]
total = con.execute("SELECT count(*) FROM companies").fetchone()[0]
print(f"companies that own property: {n_companies:,} of {total:,} ({n_companies/total:.2%})")

# the biggest owners by number of properties, with the total price paid we could see
con.execute("""
    SELECT c.company_name, c.sector,
           count(*)     AS properties,
           sum(s.value) AS total_price_paid
    FROM signals s JOIN companies c ON c.company_number = s.company_number
    WHERE s.source = 'land_registry_ccod'
    GROUP BY c.company_name, c.sector
    ORDER BY properties DESC
    LIMIT 10
""").df()

### Which sectors own the most property
A quick breakdown of the matched companies by the sector we assigned them, so we can see where property ownership concentrates.

This box shows the matched companies grouped by sector.

In [ ]:
con.execute("""
    SELECT c.sector,
           count(DISTINCT c.company_number) AS companies_owning_property
    FROM signals s JOIN companies c ON c.company_number = s.company_number
    WHERE s.source = 'land_registry_ccod'
    GROUP BY c.sector
    ORDER BY companies_owning_property DESC
""").df()

## 9. Charts
Four pictures that explain what happened: how the rows narrow down to a confident match, which method did the matching, which companies own the most, and where the properties are.

This box draws the four charts described above.

In [ ]:
import matplotlib.pyplot as plt

if len(matched) == 0:
    print("no matches to show yet. Check the Land Registry file loaded and NB05 built the full list.")
else:
    fig, ax = plt.subplots(2, 2, figsize=(13, 9))

    # A. funnel: from all owner rows down to a confident match
    funnel = {
        "owner rows\nin the file": len(owners),
        "has a reg\nnumber": int(owners["company_reg_no"].notna().sum()),
        "matched to\nour list": len(matched),
    }
    ax[0, 0].bar(list(funnel.keys()), list(funnel.values()), color="#4477aa")
    ax[0, 0].set_title("From property owners to companies matched")
    for i, v in enumerate(funnel.values()):
        ax[0, 0].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)

    # B. matches by method (each method maps to a confidence tier)
    mm = matched["method"].value_counts()
    ax[0, 1].barh(list(mm.index[::-1]), list(mm.values[::-1]), color="#228833")
    ax[0, 1].set_title("Matches by method (most reliable first)")
    for i, v in enumerate(mm.values[::-1]):
        ax[0, 1].text(v, i, f" {v:,}", va="center", fontsize=9)

    # C. the ten companies owning the most properties
    top = con.execute("""
        SELECT c.company_name, count(*) AS n
        FROM signals s JOIN companies c ON c.company_number = s.company_number
        WHERE s.source='land_registry_ccod'
        GROUP BY c.company_name ORDER BY n DESC LIMIT 10
    """).df()
    if len(top):
        ax[1, 0].barh(top["company_name"][::-1], top["n"][::-1], color="#ccbb44")
        ax[1, 0].set_title("Top companies by number of properties")
        ax[1, 0].set_xlabel("properties owned")

    # D. matched properties by region of the country
    reg = matched.assign(region=matched["region"].fillna("UNKNOWN")) \
                 .groupby("region").size().sort_values(ascending=False).head(10)
    if len(reg):
        ax[1, 1].barh(list(reg.index[::-1]), list(reg.values[::-1]), color="#ee6677")
        ax[1, 1].set_title("Matched properties by region (top 10)")
        ax[1, 1].tick_params(axis="y", labelsize=8)

    fig.suptitle("NB08 Land Registry: matching and the property signals produced", fontsize=13)
    fig.tight_layout()
    plt.show()

### Two more views: tenure and value
One chart for the freehold/leasehold split of the matched properties, and one for the companies holding the most by total price paid. Price paid is blank on many rows, so the value chart is a rough guide, not a full valuation.

This box draws the two extra charts described above.

In [ ]:
if len(matched):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

    # left: freehold vs leasehold among the matched properties
    ten = matched["tenure"].fillna("Unknown").value_counts()
    ax[0].bar(list(ten.index), list(ten.values), color="#4477aa")
    ax[0].set_title("Matched properties by tenure")
    for i, v in enumerate(ten.values):
        ax[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)

    # right: companies with the highest total price paid we could see (a rough value proxy)
    topv = con.execute("""
        SELECT c.company_name, sum(s.value) AS total
        FROM signals s JOIN companies c ON c.company_number = s.company_number
        WHERE s.source='land_registry_ccod' AND s.value IS NOT NULL
        GROUP BY c.company_name ORDER BY total DESC LIMIT 10
    """).df()
    if len(topv):
        ax[1].barh(topv["company_name"][::-1], topv["total"][::-1] / 1e6, color="#ee6677")
        ax[1].set_title("Top companies by total price paid")
        ax[1].set_xlabel("total price paid (GBP millions)")

    fig.tight_layout()
    plt.show()

This box saves and closes the database so the new records are kept.

In [ ]:
con.close()
print("saved:", DB_PATH)

## Notes and what comes next
- The registration number is the main match and is very reliable, so most of the matches here are
  exact. The name-based rows (especially `name_exact_unconfirmed`, confidence 0.6) are the ones to
  treat with care; filter on `confidence >= 0.95` if you want only the firmly matched records.
- A company can own many properties, so it can carry several `owns_property` records. That is on
  purpose: the results table stores one row per event. Group by `company_number` when you want one
  number per company.
- Coverage is limited to England and Wales (see the data-source section), so a company registered in
  Scotland or Northern Ireland only appears if it owns property here.
- Next signal source, same pattern (match, then save to `signals` with a confidence): IPO trade
  marks, which have no registration number, so they are matched by name and postcode only.